## Tentang Eksperimen

- Dataset: mimic-CXR splitted (https://huggingface.co/datasets/cchitse/mimic-cxr-with-chexbert-labels).
- Vision Encoder: BioViL-T (https://huggingface.co/microsoft/BiomedVLP-BioViL-T).
- Text Decoder: BioGPT base with 347M parameters (https://huggingface.co/docs/transformers/model_doc/biogpt).
- Vision Encoder Setup: full freeze.
- Text Decoder Setup: DAPT (PEFT-LoRA).
- Task: Report Generation.
- Specific about task:
  - input: image.
  - output 1: findings text.
  - output 2: impression text.
- Evaluation Metrics:
  - Natural Language Generation (NLG) Metrics: [BLEU-N ; N = 1, 2, 3, and 4], ROUGE-L, and METEOR. 
  - Clinical Efficacy (CE) utilizing CheXbert: precision, recall, macro F-1, and balanced accuracy.
  - CheXbert labeler for generated output for CE Metrics Evalution: https://github.com/stanfordmlgroup/CheXbert

## Tentang Arsitektur

```
+---------------------------------------------------------------------------------------+
|                              PIPELINE REPORT GENERATION                               |
+---------------------------------------------------------------------------------------+
                                        [INPUT]
                                   Gambar CXR (PIL)
                                           │
                                           ▼
+---------------------------------------------------------------------------------------+
| TAHAP PREPROCESSING CITRA (Resmi BioViL-T)                                            |
|  - Convert ke Grayscale (mode 'L') & Duplikasi ke 3 Channel (RGB)                     |
|  - Resize ke 512px (sisi terpendek)                                                   |
|  - Center Crop ke 448x448                                                             |
|  - Normalisasi sesuai statistik pretraining BioViL-T                                  |
+---------------------------------------------------------------------------------------+
                                           │
                                           ▼ [Tensor Dimensi: (B, 3, 448, 448)]
+---------------------------------------------------------------------------------------+
|  VISION ENCODER (BioViL-T) -- FULLY FROZEN (🔒)                                        |
|  ┌─────────────────────────────────────────────────────────────────────────────────┐  |
|  │  ResNet-50 Backbone (pretrained di CXR)                                         │  |
|  │      │                                                                          │  |
|  │      ▼                                                                          │  |
|  │  Patch Embeddings -> Output Dimensi: (B, 768, 14, 14)  [196 Spasial Patches]    │  |
|  └─────────────────────────────────────────────────────────────────────────────────┘  |
+---------------------------------------------------------------------------------------+
                                           │
                                           ▼
+---------------------------------------------------------------------------------------+
|  PROJECTION LAYER -- TRAINABLE (📐)                                                    |
|  ┌─────────────────────────────────────────────────────────────────────────────────┐  |
|  │  1. Adaptive Average Pooling 6x6 ──────> Dimensi: (B, 768, 6, 6)                │  |
|  │     (Kompresi informasi menjadi 36 token visual untuk efisiensi VRAM)           │  |
|  │  2. Flatten + Linear (768 -> 1024) ────> GELU Activation ───> Linear (1024)     │  |
|  │  3. Penambahan Positional Embedding (learnable)                                 │  |
|  │     Output Visual Tokens ──────────────> Dimensi: (B, 36, 1024)                 │  |
|  └─────────────────────────────────────────────────────────────────────────────────┘  |
+---------------------------------------------------------------------------------------+
                                           │
                                           ▼
+---------------------------------------------------------------------------------------+
|  TEXT TARGET PREPARATION                                                              |
|  ┌─────────────────────────────────────────────────────────────────────────────────┐  |
|  │  Format Teks: "[BOS_FINDINGS] <findings> [BOS_IMPRESSION] <impression> [EOS]"   │  |
|  │  Tokenizer (BioGPT) ───────────────────> Input IDs Dimensi: (B, L)              │  |
|  └─────────────────────────────────────────────────────────────────────────────────┘  |
+---------------------------------------------------------------------------------------+
                                           │
                                           ▼
+---------------------------------------------------------------------------------------+
|  FUSION: PREFIX CONCATENATION                                                         |
|                                                                                       |
|     [Visual Tokens (36, 1024)]   +   [Text Tokens (L, 1024)]                          |
|                 │                                │                                    |
|                 └────────────────┬───────────────┘                                    |
|                                  ▼                                                    |
|                Input Sequence Dimensi: (B, 36 + L, 1024)                          |
|                                                                                       |
|  Attention Mask : Visual Tokens = 1, Text Tokens = attention_mask asli                |
|  Target Labels  : Visual Tokens = -100 (ignored), Text Tokens = Token IDs (shifted)   |
+---------------------------------------------------------------------------------------+
                                           │
                                           ▼
+---------------------------------------------------------------------------------------+
|  TEXT DECODER (BioGPT + LoRA) -- TRAINABLE (🧠)                                        |
|  ┌─────────────────────────────────────────────────────────────────────────────────┐  |
|  │  Arsitektur           : Decoder-only Transformer (Causal LM)                     │  |
|  │  Parameter Base       : ~347M Parameters (BioGPT Base)                          │  |
|  │  LoRA Target Modules  : q_proj, v_proj (r=16, alpha=32)                          │  |
|  │  Fungsi               : Memprediksi token selanjutnya secara autoregresif       │  |
|  └─────────────────────────────────────────────────────────────────────────────────┘  |
+---------------------------------------------------------------------------------------+
                                           │
                                           ▼
+---------------------------------------------------------------------------------------+
|  LOSS COMPUTATION (🎯)                                                                |
|  - CrossEntropyLoss dieksekusi eksklusif pada token teks (posisi visual di-mask)       |
|  - Loss = Average negative log-likelihood pada target tokens                         |
+---------------------------------------------------------------------------------------+
                                           │
                                           ▼
+---------------------------------------------------------------------------------------+
|  INFERENCE (GENERATION) (🚀)                                                          |
|                                                                                       |
|  1. Inisialisasi Prompt dengan [Visual Tokens (36)] + Token [BOS_FINDINGS]            |
|  2. BioGPT melakukan dekoding token demi token secara autoregresif                    |
|  3. Proses berhenti saat token [EOS] diekstrak atau max_new_tokens tercapai          |
|  4. Teks laporan dipisahkan menggunakan token pembatas [BOS_IMPRESSION]               |
+---------------------------------------------------------------------------------------+
                                           │
                                           ▼
                                       [OUTPUT]
                        - Findings   : <deskripsi temuan medis>
                        - Impression : <kesimpulan klinis radiolog>
```

## 1. Import Library and Initial Setup

In [ ]:
# ==========================================
# standard library
# ==========================================
import gc
import json
import math
import os
import random
import re
import subprocess
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ==========================================
# natural language processing
# ==========================================
import nltk

# ==========================================
# data manipulation
# ==========================================
import numpy as np
import pandas as pd

# ==========================================
# visualization
# ==========================================
import matplotlib.pyplot as plt

# ==========================================
# deep learning
# ==========================================
import torch
import torch.nn as nn
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==========================================
# huggingface / peft
# ==========================================
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

# ==========================================
# evaluation
# ==========================================
import evaluate
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

# ==========================================
# biovil-t image encoder
# ==========================================
from health_multimodal.image import get_image_inference
from health_multimodal.image.utils import ImageModelType

# ==========================================
# suppress warnings
# ==========================================
warnings.filterwarnings("ignore")

c:\Users\Baihaqie\anaconda3\envs\ai_work\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# GPU usage setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


In [3]:
# reproducibility setup
SEED = 42
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
set_seed(SEED)

In [4]:
# global configuration dictionary
CFG = {
    # ---------- paths ----------
    'work_dir': './exp_output',
    'dapt_checkpoint': './exp_output/biogpt_dapt_adapted',
    'final_checkpoint': './exp_output/multimodal_best.pt',
    # ---------- model ----------
    'biogpt_name': 'microsoft/biogpt',
    'biovilt_transform': None,  # akan diisi otomatis
    # ---------- DAPT (text-only) ----------
    'do_dapt': True,              # SET FALSE jika ingin skip DAPT
    'dapt_epochs': 2,
    'dapt_batch_size': 8,
    'dapt_lr': 2e-4,
    'dapt_max_len': 384,
    # ---------- LoRA ----------
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'lora_target': ['q_proj', 'v_proj'],  # layer attention di BioGPT
    # ---------- multimodal ----------
    'mm_batch_size': 4,
    'grad_accum': 4,              # Effective batch = 4 * 4 = 16
    'mm_epochs': 5,
    'mm_lr': 2e-5,                # Learning rate untuk LoRA + Proyeksi
    'warmup_ratio': 0.1,
    'max_grad_norm': 1.0,
    # ---------- visual architecture ----------
    'pool_grid': 6,               # 6x6 = 36 visual token (hemat VRAM)
    'vision_dim': 768,            # Output BioViLT patch (akan dicek otomatis)
    'text_dim': 1024,             # Hidden size BioGPT (akan dicek otomatis)
    # ---------- inference ----------
    'max_new_tokens': 200,
    'num_beams': 4,
    # ---------- evaluation ----------
    'chexpert_conditions': [
        'Atelectasis', 
        'Cardiomegaly', 
        'Consolidation', 
        'Edema',
        'Enlarged Cardiomediastinum', 
        'Fracture', 
        'Lung Lesion',
        'Lung Opacity', 
        'No Finding', 
        'Pleural Effusion', 
        'Pleural Other',
        'Pneumonia', 
        'Pneumothorax', 
        'Support Devices'
    ],
}

Path(CFG['work_dir']).mkdir(parents=True, exist_ok=True)
print(f"Work directory: {CFG['work_dir']}")
print(json.dumps(CFG, indent=2, default=str))

Work directory: ./exp_output
{
  "work_dir": "./exp_output",
  "dapt_checkpoint": "./exp_output/biogpt_dapt_adapted",
  "final_checkpoint": "./exp_output/multimodal_best.pt",
  "biogpt_name": "microsoft/biogpt",
  "biovilt_transform": null,
  "do_dapt": true,
  "dapt_epochs": 2,
  "dapt_batch_size": 8,
  "dapt_lr": 0.0002,
  "dapt_max_len": 384,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "lora_target": [
    "q_proj",
    "v_proj"
  ],
  "mm_batch_size": 4,
  "grad_accum": 4,
  "mm_epochs": 5,
  "mm_lr": 2e-05,
  "warmup_ratio": 0.1,
  "max_grad_norm": 1.0,
  "pool_grid": 6,
  "vision_dim": 768,
  "text_dim": 1024,
  "max_new_tokens": 200,
  "num_beams": 4,
  "chexpert_conditions": [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Enlarged Cardiomediastinum",
    "Fracture",
    "Lung Lesion",
    "Lung Opacity",
    "No Finding",
    "Pleural Effusion",
    "Pleural Other",
    "Pneumonia",
    "Pneumothorax",
    "Support Devices"
  ]


## 2. Load Data

In [5]:
# load dataset
dataset = load_dataset("cchitse/mimic-cxr-with-chexbert-labels")

In [6]:
# preview structure of dataset
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'findings', 'impression', '__index_level_0__', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['image', 'findings', 'impression', '__index_level_0__', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['image', 'findings', 'impression', '__index_level_0__', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural 

In [7]:
# get sample to verify the content of dataset
print("Image type :", type(dataset['train'][0]['image']))
print("Findings   :", dataset['train'][0]['findings'][:120], '...')
print("Impression :", dataset['train'][0]['impression'][:120], '...')

Image type : <class 'PIL.JpegImagePlugin.JpegImageFile'>
Findings   : Dobhoff tube now ends in the proximal stomach. Stable, mild cardiomegaly. Unchanged moderate right pleural effusion and  ...
Impression : Dobhoff tube now ends in the proximal stomach. Unchanged moderate right pleural effusion and moderate to large left pleu ...


## 3. Pre-processing

In [8]:
# load BioGPT tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG['biogpt_name'])

# if BioGPT doesn't have pad_token, set to eos
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# add special token which is a splitter so that get tidy output
SPECIAL_TOKENS = ['[BOS_FINDINGS]', '[BOS_IMPRESSION]', '[REDACTED]']
tokenizer.add_special_tokens({'additional_special_tokens': SPECIAL_TOKENS})

BOS_FINDINGS_ID = tokenizer.convert_tokens_to_ids('[BOS_FINDINGS]')
BOS_IMPRESSION_ID = tokenizer.convert_tokens_to_ids('[BOS_IMPRESSION]')
print(f"special token IDs: BOS_FINDINGS={BOS_FINDINGS_ID}, BOS_IMPRESSION={BOS_IMPRESSION_ID}")

special token IDs: BOS_FINDINGS=42384, BOS_IMPRESSION=42385


In [9]:
def clean_text(text: str) -> str:
    """clean MIMIC text: handle NaN, replace with underline, normalize whitespace."""
    if text is None or pd.isna(text):
        return ''
    text = re.sub(r'_+', '[REDACTED]', text)  # de-identifikasi
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def build_report_text(findings: str, impression: str) -> str:
    """format target: [BOS_FINDINGS] ... [BOS_IMPRESSION] ... [EOS]"""
    f = clean_text(findings)
    i = clean_text(impression)
    return f"[BOS_FINDINGS] {f} [BOS_IMPRESSION] {i} {tokenizer.eos_token}"

# create DataFrame of text for DAPT
def prepare_text_df(hf_split):
    df = pd.DataFrame({
        'findings': hf_split['findings'],
        'impression': hf_split['impression']
    })
    df['clean_findings'] = df['findings'].apply(clean_text)
    df['clean_impression'] = df['impression'].apply(clean_text)
    # filter empty row
    mask = (df['clean_findings'].str.len() > 0) & (df['clean_impression'].str.len() > 0)
    df = df[mask].reset_index(drop=True)
    df['full_text'] = df.apply(lambda r: build_report_text(r['clean_findings'], r['clean_impression']), axis=1)
    return df

train_text_df = prepare_text_df(dataset['train'])
val_text_df = prepare_text_df(dataset['validation'])
test_text_df = prepare_text_df(dataset['test'])
print(f"Train: {len(train_text_df)} | Val: {len(val_text_df)} | Test: {len(test_text_df)}")

Train: 15997 | Val: 2000 | Test: 2000


## 4. Stage-1: DAPT (Domain-Adaptive Pre-training) for BioGPT

In [ ]:
class DAPTDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        input_ids = enc['input_ids'].squeeze(0)
        attn_mask = enc['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[attn_mask == 0] = -100  # abaikan padding
        return {'input_ids': input_ids, 'attention_mask': attn_mask, 'labels': labels}

# set LoRA
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    target_modules=CFG['lora_target'],
    bias='none',
)

if CFG['do_dapt']:
    print(">>> start DAPT (Text-only) <<<")

    dapt_train = DAPTDataset(train_text_df['full_text'], tokenizer, CFG['dapt_max_len'])
    dapt_val = DAPTDataset(val_text_df['full_text'], tokenizer, CFG['dapt_max_len'])

    dapt_loader = DataLoader(dapt_train, batch_size=CFG['dapt_batch_size'], shuffle=True, num_workers=0, pin_memory=True)
    dapt_val_loader = DataLoader(dapt_val, batch_size=CFG['dapt_batch_size'], shuffle=False, num_workers=0, pin_memory=True)

    # Load BioGPT base + resize embedding (karena special token baru)
    dapt_model = AutoModelForCausalLM.from_pretrained(CFG['biogpt_name'])
    dapt_model.resize_token_embeddings(len(tokenizer))

    dapt_model = get_peft_model(dapt_model, lora_cfg)
    dapt_model.to(device)
    dapt_model.print_trainable_parameters()

    optimizer = AdamW(dapt_model.parameters(), lr=CFG['dapt_lr'])
    total_steps = len(dapt_loader) * CFG['dapt_epochs']
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )

    best_val_loss = float('inf')

    for epoch in range(CFG['dapt_epochs']):
        # ---------- TRAINING ----------
        dapt_model.train()
        total_loss = 0
        pbar = tqdm(dapt_loader, desc=f"DAPT Epoch {epoch+1}/{CFG['dapt_epochs']}")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = dapt_model(**batch)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(dapt_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})

        avg_train_loss = total_loss / len(dapt_loader)
        print(f"DAPT Epoch {epoch+1} | Avg Train Loss: {avg_train_loss:.4f}")

        # ---------- VALIDATION ----------
        dapt_model.eval()
        val_loss = 0
        with torch.no_grad():
            for val_batch in tqdm(dapt_val_loader, desc="Validating", leave=False):
                val_batch = {k: v.to(device) for k, v in val_batch.items()}
                val_outputs = dapt_model(**val_batch)
                val_loss += val_outputs.loss.item()
        avg_val_loss = val_loss / len(dapt_val_loader)
        print(f"DAPT Epoch {epoch+1} | Avg Val Loss: {avg_val_loss:.4f}")

        # ---------- SAVE CHECKPOINT BASED ON VALIDATION LOSS ----------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            dapt_model.save_pretrained(CFG['dapt_checkpoint'])
            tokenizer.save_pretrained(CFG['dapt_checkpoint'])
            print(f"  -> Best DAPT model saved (val_loss={best_val_loss:.4f})")

    # clean memory
    del dapt_model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()
    print("DAPT Selesai.\n")
else:
    print(">>> DAPT dilewati (CFG['do_dapt']=False) <<<")

>>> start DAPT (Text-only) <<<


pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

trainable params: 1,572,864 || all params: 348,339,200 || trainable%: 0.4515


DAPT Epoch 1/2:   0%|          | 0/2000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Load Vision Encoder and Define Model Architecture

In [ ]:
# load BioViL-T Image Encoder
image_engine = get_image_inference(ImageModelType.BIOVIL_T)
biovilt_transform = image_engine.transform
image_engine.to(device)
image_engine.eval()

In [ ]:
# total freeze layers for vision encoders
for param in image_engine.model.parameters():
    param.requires_grad = False

In [ ]:
# check dimention with dummy
sample_img = dataset['train'][0]['image']
if sample_img.mode != 'L':
    sample_img = sample_img.convert('L')
real_pixel_values = biovilt_transform(sample_img).unsqueeze(0).to(device)  # (1, C, H, W)

with torch.no_grad():
    dummy_out = image_engine.model(real_pixel_values)
    vision_dim = dummy_out.patch_embeddings.shape[-1]
    print(f"Vision hidden dimension from real sample: {vision_dim}")
    CFG['vision_dim'] = vision_dim

In [ ]:
# check dimention of BioGPT
text_decoder_base = AutoModelForCausalLM.from_pretrained(CFG['biogpt_name'])
text_dim = text_decoder_base.config.hidden_size
CFG['text_dim'] = text_dim
print(f"BioGPT hidden dimension: {text_dim}")
del text_decoder_base
gc.collect()

In [ ]:
# define multimodal
class VisualProjection(nn.Module):
    """Proyeksi patch 768-dim -> 36 token 1024-dim with Adaptive Pooling."""
    def __init__(self, in_dim, out_dim, grid_size=6):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((grid_size, grid_size))
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim),
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, grid_size * grid_size, out_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, patch_emb):
        # patch_emb: (B, C, H, W) -> pool -> (B, C, grid, grid)
        pooled = self.pool(patch_emb)
        B, C, H, W = pooled.shape
        tokens = pooled.flatten(2).transpose(1, 2)  # (B, grid*grid, C)
        tokens = self.proj(tokens)  # (B, 36, out_dim)
        return tokens + self.pos_embed

class ReportGenerator(nn.Module):
    def __init__(self, vision_encoder, projection, decoder):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.projection = projection
        self.decoder = decoder

    def encode_image(self, pixel_values):
        with torch.no_grad():
            out = self.vision_encoder(pixel_values)
        return out.patch_embeddings

    def forward(self, pixel_values, input_ids, attention_mask, labels=None):
        # visual extraction and projection
        patch_emb = self.encode_image(pixel_values)
        visual_tokens = self.projection(patch_emb)

        # text embedding
        text_embeds = self.decoder.get_input_embeddings()(input_ids)

        # concatenate
        inputs_embeds = torch.cat([visual_tokens, text_embeds], dim=1)
        vis_mask = torch.ones(visual_tokens.shape[0], visual_tokens.shape[1], device=device, dtype=torch.long)
        full_attn_mask = torch.cat([vis_mask, attention_mask], dim=1)

        # labels
        vis_labels = torch.full((visual_tokens.shape[0], visual_tokens.shape[1]), -100, device=device, dtype=torch.long)
        full_labels = torch.cat([vis_labels, labels], dim=1) if labels is not None else None

        # forward to decoder without labels
        outputs = self.decoder(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attn_mask,
            return_dict=True
        )
        logits = outputs.logits

        # calculate loss with label smoothing
        loss = None
        if full_labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = full_labels[..., 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)
            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )
        return {'loss': loss, 'logits': logits}

    @torch.no_grad()
    def generate(self, pixel_values, **gen_kwargs):
        """generate text from image"""
        patch_emb = self.encode_image(pixel_values)
        visual_tokens = self.projection(patch_emb)

        start_token = torch.tensor([[BOS_FINDINGS_ID]], device=device)
        start_embeds = self.decoder.get_input_embeddings()(start_token)

        inputs_embeds = torch.cat([visual_tokens, start_embeds], dim=1)
        vis_mask = torch.ones(visual_tokens.shape[0], visual_tokens.shape[1], device=device, dtype=torch.long)
        attn_mask = torch.cat([vis_mask, torch.ones((visual_tokens.shape[0], 1), device=device, dtype=torch.long)], dim=1)

        outputs = self.decoder.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attn_mask,
            **gen_kwargs
        )
        return outputs

## 6. Load Decoder (DAPT result) and Initialize Final Model

In [ ]:
# load decoder + LoRA + final model
if CFG['do_dapt'] and Path(CFG['dapt_checkpoint']).exists():
    print(f"Memuat adapter LoRA hasil DAPT dari {CFG['dapt_checkpoint']}")
    base_decoder = AutoModelForCausalLM.from_pretrained(CFG['biogpt_name'])
    base_decoder.resize_token_embeddings(len(tokenizer))
    decoder = PeftModel.from_pretrained(base_decoder, CFG['dapt_checkpoint'])
else:
    print("Memuat BioGPT base + LoRA baru (tanpa DAPT)")
    base_decoder = AutoModelForCausalLM.from_pretrained(CFG['biogpt_name'])
    base_decoder.resize_token_embeddings(len(tokenizer))
    decoder = get_peft_model(base_decoder, lora_cfg)

decoder.to(device)
decoder.print_trainable_parameters()

# initialize projection
projection = VisualProjection(
    in_dim=CFG['vision_dim'],
    out_dim=CFG['text_dim'],
    grid_size=CFG['pool_grid']
).to(device)

# final model
model = ReportGenerator(
    vision_encoder=image_engine.model,
    projection=projection,
    decoder=decoder
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total: {total_params:,} | Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

## 7. Define Multimodal Dataset and DataLoader

In [ ]:
class MultimodalCXRDataset(Dataset):
    def __init__(self, hf_split, tokenizer, transform, max_len):
        self.hf_split = hf_split
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len = max_len
        # valid index
        self.valid_indices = []
        for i in range(len(hf_split)):
            f = clean_text(hf_split[i]['findings'])
            i_text = clean_text(hf_split[i]['impression'])
            if len(f) > 0 and len(i_text) > 0:
                self.valid_indices.append(i)
        print(f"Valid samples: {len(self.valid_indices)}/{len(hf_split)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.hf_split[self.valid_indices[idx]]

        # image
        img = sample['image']
        if img.mode != 'L':
            img = img.convert('L')
        pixel_values = self.transform(img)  # (1, H, W) atau (C, H, W)

        # text
        text = build_report_text(sample['findings'], sample['impression'])
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )
        input_ids = enc['input_ids'].squeeze(0)
        attn_mask = enc['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[attn_mask == 0] = -100

        return {
            'pixel_values': pixel_values,
            'input_ids': input_ids,
            'attention_mask': attn_mask,
            'labels': labels,
        }


MAX_LEN = 512
print(f"max sequence length: {MAX_LEN}")
train_mm = MultimodalCXRDataset(dataset['train'], tokenizer, biovilt_transform, MAX_LEN)
val_mm = MultimodalCXRDataset(dataset['validation'], tokenizer, biovilt_transform, MAX_LEN)

def collate_fn(batch):
    pixel_values = torch.stack([b['pixel_values'] for b in batch])
    input_ids = torch.stack([b['input_ids'] for b in batch])
    attention_mask = torch.stack([b['attention_mask'] for b in batch])
    labels = torch.stack([b['labels'] for b in batch])
    return {
        'pixel_values': pixel_values,
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }

train_loader = DataLoader(train_mm, batch_size=CFG['mm_batch_size'], shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_mm, batch_size=CFG['mm_batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 8. Define Training Loop Multimodal

In [ ]:
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG['mm_lr'],
    weight_decay=0.01
)

total_steps = math.ceil(len(train_loader) / CFG['grad_accum']) * CFG['mm_epochs']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(CFG['warmup_ratio'] * total_steps),
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler()
best_val_loss = float('inf')

# list to track and save loss history
train_losses = []
val_losses = []

for epoch in range(CFG['mm_epochs']):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['mm_epochs']}")

    for step, batch in enumerate(pbar):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.cuda.amp.autocast():
            outputs = model(**batch)
            loss = outputs.loss / CFG['grad_accum']

        scaler.scale(loss).backward()

        if (step + 1) % CFG['grad_accum'] == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * CFG['grad_accum']
        pbar.set_postfix({'loss': loss.item() * CFG['grad_accum']})

    avg_train_loss = total_loss / len(train_loader)

    # validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast():
                out = model(**batch)
                val_loss += out.loss.item()
    avg_val_loss = val_loss / len(val_loader)

    # save loss history
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'val_loss': avg_val_loss,
        }, CFG['final_checkpoint'])
        print(f"  -> Best model saved (val_loss={avg_val_loss:.4f})")

print(f"\nTraining Finished! Best Val Loss: {best_val_loss:.4f}")

## 9. Inference $-$ Generate Report

In [ ]:
# load best checkpoint
checkpoint = torch.load(CFG['final_checkpoint'], map_location=device)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']} (val_loss={checkpoint['val_loss']:.4f})")

def split_report(text: str) -> Tuple[str, str]:
    """split Findings and Impression from output model."""
    text = text.replace(tokenizer.eos_token, '').strip()
    if '[BOS_IMPRESSION]' in text:
        parts = text.split('[BOS_IMPRESSION]')
        findings = parts[0].replace('[BOS_FINDINGS]', '').strip()
        impression = parts[1].strip() if len(parts) > 1 else ''
    else:
        findings = text.replace('[BOS_FINDINGS]', '').strip()
        impression = ''
    return findings, impression

@torch.no_grad()
def generate_report(pil_image: Image.Image) -> Dict[str, str]:
    """generate findings and impression from PIL Image."""

    # preprocess
    if pil_image.mode != 'L':
        pil_image = pil_image.convert('L')
    pixel_values = biovilt_transform(pil_image).unsqueeze(0).to(device)

    # generate
    gen_kwargs = {
        'max_new_tokens': CFG['max_new_tokens'],
        'num_beams': CFG['num_beams'],
        'early_stopping': True,
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'no_repeat_ngram_size': 3,
    }
    gen_ids = model.generate(pixel_values, **gen_kwargs)

    # if exist, cut visual prefix
    n_visual = CFG['pool_grid'] ** 2  # 36
    # panjang output > n_visual dan token ke-n_visual adalah BOS_FINDINGS
    if gen_ids.shape[1] > n_visual and gen_ids[0, n_visual] == BOS_FINDINGS_ID:
        gen_ids = gen_ids[:, n_visual:]  # potong 36 token pertama

    text = tokenizer.decode(gen_ids[0], skip_special_tokens=False)
    findings, impression = split_report(text)
    return {'findings': findings, 'impression': impression}

In [ ]:
# quick test
result = generate_report(dataset['test'][0]['image'])
print("sample inference:")
print(f"Findings: {result['findings'][:150]}...")
print(f"Impression: {result['impression'][:150]}...")

## 10. Evaluation: NLG + CE via CheXbert

In [ ]:
# inference on test set
# test_subset = dataset['test'].select(range(n)) jika mau ambil n sampel saja
test_subset = dataset['test']
results = []
for sample in tqdm(test_subset, desc="Inference Test Set"):
    pred = generate_report(sample['image'])
    results.append({
        'gt_findings': clean_text(sample['findings']),
        'gt_impression': clean_text(sample['impression']),
        'pred_findings': pred['findings'],
        'pred_impression': pred['impression'],
    })
df = pd.DataFrame(results)
df.to_csv(f"{CFG['work_dir']}/test_predictions.csv", index=False)

In [ ]:
# visualize loss history
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(CFG['work_dir'], 'loss_plot.png'), dpi=150)
plt.show()
print(f"Loss plot saved to {CFG['work_dir']}/loss_plot.png")

In [ ]:
# NLG metrics
bleu = evaluate.load('sacrebleu')
rouge = evaluate.load('rouge')
meteor = evaluate.load('meteor')

def compute_nlg(preds, refs, name):
    refs_wrap = [[r] for r in refs]
    scores = {}
    for n in [1,2,3,4]:
        scores[f'BLEU-{n}'] = bleu.compute(predictions=preds, references=refs_wrap, max_order=n)['score']
    scores['ROUGE-L'] = rouge.compute(predictions=preds, references=refs)['rougeL']
    scores['METEOR'] = meteor.compute(predictions=preds, references=refs)['meteor']
    print(f"\n=== {name} ===")
    for k,v in scores.items():
        print(f"  {k}: {v:.4f}")
    return scores

findings_metrics = compute_nlg(df['pred_findings'].tolist(), df['gt_findings'].tolist(), "Findings")
impression_metrics = compute_nlg(df['pred_impression'].tolist(), df['gt_impression'].tolist(), "Impression")

nlg_df = pd.DataFrame([
    {'section': 'Findings', **findings_metrics},
    {'section': 'Impression', **impression_metrics}
])
nlg_df.to_csv(os.path.join(CFG['work_dir'], 'nlg_metrics.csv'), index=False)
print("NLG metrics saved.")

In [ ]:
# path and setup
CHEXBERT_DIR = './CheXbert'
CHEXBERT_CKPT = './CheXbert/pretrained_weights/chexbert.pth'

# clone repo if does not exist on your directory
if not os.path.exists(CHEXBERT_DIR):
    print("Cloning CheXbert repository...")
    subprocess.run(['git', 'clone', 'https://github.com/stanfordmlgroup/CheXbert.git', CHEXBERT_DIR], check=True)
    print("CheXbert cloned.")
else:
    print("CheXbert directory already exists.")

# install dependencies of CheXbert
req_file = os.path.join(CHEXBERT_DIR, 'requirements.txt')
if os.path.exists(req_file):
    print("installing CheXbert dependencies...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req_file, '-q'], check=True)
    print("dependencies installed.")

# CheXbert checkpoint
if not os.path.exists(CHEXBERT_CKPT):
    raise FileNotFoundError(f"Checkpoint tidak ditemukan di {CHEXBERT_CKPT}")
print("CheXbert checkpoint ditemukan.")

# helper function: prepare for CheXbert input
def prepare_chexbert_input(texts: List[str], output_csv: str):
    """save text in a format required by the CheXbert."""
    pd.DataFrame({'Report Impression': texts}).to_csv(output_csv, index=False)
    return output_csv

def run_chexbert_labeling(input_csv: str, output_dir: str) -> pd.DataFrame:
    """run CheXbert labeler via subprocess."""
    label_script = os.path.join(CHEXBERT_DIR, 'src', 'label.py')
    os.makedirs(output_dir, exist_ok=True)
    cmd = [
        'python', label_script,
        '-d', input_csv,
        '-o', output_dir,
        '-c', CHEXBERT_CKPT
    ]
    print(f"running CheXbert: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)
    labeled_path = os.path.join(output_dir, 'labeled_reports.csv')
    return pd.read_csv(labeled_path)

def binarize_chexbert_labels(df: pd.DataFrame, conditions: List[str]) -> np.ndarray:
    """
    binarize CheXbert label:
    1 (positive) -> 1, rest (0, -1, NaN) -> 0.
    """
    binary = []
    for cond in conditions:
        if cond in df.columns:
            col = df[cond].fillna(0).values
            binary.append((col == 1).astype(int))
        else:
            binary.append(np.zeros(len(df), dtype=int))
    return np.array(binary).T  # (n_samples, n_conditions)

def binarize_ground_truth_labels(hf_dataset, conditions: List[str]) -> np.ndarray:
    """
    binarize ground truth label from  HuggingFace dataset.
    """
    binary = []
    for cond in conditions:
        # take a column from dataset
        col = hf_dataset[cond]
        # convert to numpy, NaN -> 0, then binarize (1 -> 1, rest -> 0)
        col_array = np.array(col, dtype=np.float32)
        col_array = np.nan_to_num(col_array, nan=0)
        binary.append((col_array == 1).astype(int))
    return np.array(binary).T  # (n_samples, n_conditions)

In [ ]:
# define chexpert conditions
conditions = CFG['chexpert_conditions']

# ground truth from dataset['test']
gt_binary = binarize_ground_truth_labels(dataset['test'], conditions)

# display ground truth shape
print(f"ground truth shape: {gt_binary.shape}")

# prepare predicted text for CheXbert
pred_full_text = (df['pred_findings'] + ' ' + df['pred_impression']).tolist()
work_dir = CFG['work_dir']
pred_input_csv = prepare_chexbert_input(pred_full_text, os.path.join(work_dir, 'chexbert_pred_input.csv'))

# run CheXbert for predicted text only
pred_labels_df = run_chexbert_labeling(pred_input_csv, os.path.join(work_dir, 'chexbert_pred_output'))

# binarize the CheXbert prediction
pred_binary = binarize_chexbert_labels(pred_labels_df, conditions)
print(f"prediction labels shape: {pred_binary.shape}")

# verify that number of samples equal
assert gt_binary.shape[0] == pred_binary.shape[0], "number of samples ground truth and predicted is not equal!"

# calculate CE metrics per condition
ce_results = []
for i, cond in enumerate(conditions):
    y_true = gt_binary[:, i]
    y_pred = pred_binary[:, i]
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    ce_results.append({
        'condition': cond,
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f1': round(f1, 4),
        'balanced_accuracy': round(bal_acc, 4),
        'support': int(y_true.sum())
    })

# calculate CE metrics overall
macro_prec = np.mean([r['precision'] for r in ce_results])
macro_rec = np.mean([r['recall'] for r in ce_results])
macro_f1 = np.mean([r['f1'] for r in ce_results])
macro_bal_acc = np.mean([r['balanced_accuracy'] for r in ce_results])

ce_results.append({
    'condition': 'MACRO_AVERAGE',
    'precision': round(macro_prec, 4),
    'recall': round(macro_rec, 4),
    'f1': round(macro_f1, 4),
    'balanced_accuracy': round(macro_bal_acc, 4),
    'support': '-'
})
ce_df = pd.DataFrame(ce_results)

# display and save
print("\n" + "="*60)
print("CLINICAL EFFICACY METRICS (via CheXbert)")
print("="*60)
print(ce_df.to_string(index=False))

print("\nMACRO AVERAGE:")
print(f"  Precision         : {macro_prec:.4f}")
print(f"  Recall            : {macro_rec:.4f}")
print(f"  F1                : {macro_f1:.4f}")
print(f"  Balanced Accuracy : {macro_bal_acc:.4f}")

# save to CSV
ce_df.to_csv(os.path.join(work_dir, 'clinical_efficacy_metrics.csv'), index=False)
print(f"\nCE results saved to {work_dir}/clinical_efficacy_metrics.csv")